In [ ]:
import ixmp4
import pyam
import nomenclature

In [ ]:
df = pyam.concat(
    [
        "raw/Engage2.0_data/ENGAGE_scenario_data_world_r2.0.csv",
        "raw/Engage2.0_data/ENGAGE_scenario_data_r5_regions_r2.0.csv",
        "raw/Engage2.0_data/ENGAGE_scenario_data_r10_regions_r2.0.csv",
    ]
)

In [ ]:
df.scenario

In [ ]:
#definition = nomenclature.DataStructureDefinition("../definitions/")
definition = nomenclature.DataStructureDefinition("../../common-definitions/definitions/")

In [ ]:
validation_args = ["upper_bound", "lower_bound", "value", "rtol", "atol", "range"]

validation_list = list()

for name, variable in definition.variable.items():
    if any([i in validation_args for i in variable.extra_attributes]):
        validation_list.append(
            dict(
                variable=name,
                validation=[dict([(key, value) for key, value in variable.extra_attributes.items() if key in validation_args])]
            )
        )

In [ ]:
validator = nomenclature.processor.DataValidator(criteria_items=validation_list, file=".")

In [ ]:
validator.apply(df)

In [ ]:
dict(
    ("variable", name), 
    [(key, value) for key, value in variable.extra_attributes.items() if key in validation_args])

In [ ]:
dict([("variable", name)]+ [(key, value) for key, value in variable.extra_attributes.items() if key in validation_args])

In [ ]:
validation_list

In [ ]:
# rename scenarios for clear reference to ENGAGE
df.rename(
    scenario=dict([(i, "ENGAGE-" + i[3:].replace("_", "-")) for i in df.scenario]),
    inplace=True,
)

In [ ]:
# remove variables of little relevance that are not included in common-definitions
df.filter(
    variable=[
        "*AR5 climate diagnostics*",
        "Diagnostics|MAGICC6*",
        "Carbon Sequestration|Other",
        "Secondary Energy",
        "Food Energy Supply",
        "Investment|Energy Supply|Electricity|Non-fossil",
        "Investment|Energy Supply|Extraction|Bioenergy",
        "Investment|Energy Supply|Hydrogen|Renewable",
        "Policy Cost|Consumption Loss",
        "Policy Cost|GDP Loss",
        "Policy Cost|Area under MAC Curve",
        "Price|Agriculture|Non-Energy Crops and Livestock|Index",
        "Yield|Cereal",
        "Policy Cost|Additional Total Energy System Cost",
        "Carbon Sequestration|CCS|Biomass|Energy|Demand|Industry", 
        "Final Energy|Residential and Commercial|Solids|Biomass|Traditional",
        "Final Energy|Transportation|Liquids|Natural Gas",
        "Capacity Additions|Electricity|Storage Capacity",
        "Food Demand",
        "Food Demand|Crops",
        "Food Demand|Livestock",
        "Carbon Sequestration|CCS", # maybe move to mapping???
    ],
    keep=False,
    inplace=True
)

In [ ]:
# rename units
df.rename(
    unit={
        "US$2010/kW OR local currency/kW": "USD_2010/kW",
        "US$2010/kW": "USD_2010/kW",
        "billion US$2010/yr": "billion USD_2010/yr",
        "billion US$2010/yr OR local currency/yr": "billion USD_2010/yr",
        "billion US$2010/yr or local currency/yr": "billion USD_2010/yr",
        "US$2010/t CO2": "USD_2010/t CO2",
        "US$2010/tCO2": "USD_2010/t CO2",
        "US$2010/t CO2 or local currency/t CO2": "USD_2010/t CO2",
        "million Ha/yr": "million ha",
        "Million": "million",
        "Mt NOx/yr": "Mt NO2/yr",  
        "Mt N2O/yr": "kt N2O/yr",
        "million m3/yr": "km3/yr",
    },
    inplace=True,
)

In [ ]:
# update carbon-management variables
carbon_management_mapping = {
#    "Carbon Sequestration|CCS": "Carbon Capture|Geological Storage",
    "Carbon Sequestration|CCS|Biomass": "Carbon Capture|Geological Storage|Biomass",
    "Carbon Sequestration|CCS|Biomass|Energy|Supply": "Carbon Capture|Energy|Supply|Biomass",
    "Carbon Sequestration|CCS|Fossil": "Carbon Capture|Energy|Fossil",
    "Carbon Sequestration|CCS|Fossil|Energy|Demand|Industry": "Carbon Capture|Energy|Demand|Industry",
    "Carbon Sequestration|CCS|Fossil|Energy|Supply": "Carbon Capture|Energy|Supply|Fossil",
    "Carbon Sequestration|CCS|Industrial Processes": "Carbon Capture|Industrial Processes",
    "Carbon Sequestration|Land Use|Afforestation": "Carbon Removal|Land Use|Re/Afforestation",
    "Carbon Sequestration|Direct Air Capture": "Carbon Removal|Geological Storage|Direct Air Capture",
    "Carbon Sequestration|Enhanced Weathering": "Carbon Removal|Enhanced Weathering",
    "Carbon Sequestration|Land Use": "Carbon Removal|Land Use",
}

df.rename(variable=carbon_management_mapping, inplace=True)

In [ ]:
# fix unit issue in WITCH
df.rename(
    variable={"Final Energy|Non-Energy Use": "Final Energy|Non-Energy Use"},
    unit={"": "EJ/yr"},
    inplace=True,
)

In [ ]:
project = "engage"
legacy_mapping = {}

for code, attrs in definition.variable.items():
    if project in attrs.extra_attributes:
        legacy_mapping[attrs.__getattr__(project)] = code

df.rename(variable=legacy_mapping, inplace=True)

In [ ]:
df.filter(variable="Capacity Additions|Electricity|*", unit="GW", keep=False, inplace=True)

In [ ]:
df.rename(
    region={
        "China (R10)": "China+ (R10)",
        "European Union and Western Europe (R10)": "Europe (R10)",
        "Other Asian countries (R10)": "Rest of Asia (R10)",
        "South Asia (R10)": "India+ (R10)",
        "Sub-Saharan Africa (R10)": "Africa (R10)",
    },
    inplace=True,
)

In [ ]:
definition.validate(df)

In [ ]:
df.set_meta("H2020 ENGAGE", "Project")
df.set_meta("Riahi et al. (2021)", "Scientific Manuscript (Citation)")
df.set_meta("10.1038/s41558-021-01215-2", "Scientific Manuscript (DOI)")
df.set_meta("10.5281/zenodo.5553976", "Data Source (DOI)")

In [ ]:
df.model

In [ ]:
for model in [
#    'MESSAGEix-GLOBIOM 1.1',
#    'POLES-JRC ENGAGE',
#    'REMIND-MAgPIE 2.1-4.2',
#    'TIAM-ECN 1.1',
#    'WITCH 5.0'
]:
    df.filter(model=model).to_ixmp4("scenariocompass-migration")
    print(model)

In [ ]:
df.to_ixmp4("scenariocompass-migration")

In [ ]:
## Fixing stuff

In [ ]:
platform = ixmp4.Platform("scenariocompass-migration")

In [ ]:
data = platform.iamc.tabulate(variable="Investment|Energy Supply|Electricity|Storage")

In [ ]:
df_correct = pyam.IamDataFrame(data)

In [ ]:
df_correct

In [ ]:
data = platform.iamc.tabulate(variable="Carbon Removal|Geological Storage|Biomass")

In [ ]:
df_wrong = pyam.IamDataFrame(data)

In [ ]:
df_wrong.meta

In [ ]:
for model, scenario in df_wrong.index:
    run = platform.runs.get(model, scenario)
    data = run.iamc.tabulate(variable="Carbon Removal|Geological Storage|Biomass")
    run.iamc.remove(data)
    data["variable"] = "Carbon Capture|Geological Storage|Biomass"
    run.iamc.add(data)

In [ ]:
    run.iamc.remove(data)
    data["variable"] = "Carbon Capture|Geological Storage|Biomass"
    run.iamc.add(data)

In [ ]:
for run in runs:
    data = run.iamc.tabulate(variable="Carbon Removal|Geological Storage")
    if not data.empty:
        run.iamc.remove(data)

In [ ]:
runs = platform.runs.list(default_only=False, is_default=False)

In [ ]:
runs

In [ ]:
for run in runs:
    _df = df.filter(model=run.model.name, scenario=run.scenario.name)
    run.iamc.add(_df.data)
    run.meta = dict(_df.meta.loc[(run.model.name, run.scenario.name)])
    run.set_as_default()

In [ ]:
for model, scenario in df.index:
    try:
        platform.runs.get(model=model, scenario=scenario)
    except:
        df.filter(model=model, scenario=scenario).to_ixmp4("scenariocompass-migration")